# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and explore the [FAIR²: Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

**Dataset Source**  
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

---

In [ ]:
# Install mlcroissant (will have no effect if already installed)
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(metadata.description)

## 2. Data Overview
Inspect the available record sets, their `@id`s, and the fields they contain.

All references to entities (record sets, fields, columns) will use their `@id` for precise identification.

In [ ]:
# List available record sets and their fields by @id
print("Available record sets and fields (@id):\n")

# Get the record set metadata from the dataset
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata. The dataset may define its record sets in the distributions or via datafiles.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.id}")
        for field in rs.fields:
            print(f"  - Field: {field.id} (type: {field.data_type})")
        print("")

# As an alternative, try to access via dataset.record_sets property if no record set in main metadata
# If record sets are not present, try to infer from distributions
if not record_sets and hasattr(dataset, 'record_sets'):
    inferred_record_sets = dataset.record_sets
    for rs in inferred_record_sets:
        print(f"Record Set: {rs.id}")
        for field in rs.fields:
            print(f"  - Field: {field.id} (type: {field.data_type})")

Next, let's get a preview of one of the record sets. We'll print a few records with all field values, referenced by their `@id`.

In [ ]:
# Get the actual record set @id(s) for use with dataset.records()
# If available, list and select the first for demonstration
actual_record_sets = getattr(metadata, 'record_sets', [])

if len(actual_record_sets) > 0:
    # Take the first record set for preview
    record_set_id = actual_record_sets[0].id
    print(f"Previewing records for record set: {record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No record sets found in metadata for preview. Please check the metadata's 'distribution' or 'recordSet' fields.")

## 3. Data Extraction
We'll load all available record sets into pandas DataFrames using their `@id` fields.

Entities in the schema are always referenced by their unique `@id`. This enables reproducible programmatic access.

In [ ]:
# Collect all record_set @id values
record_sets = getattr(metadata, 'record_sets', [])

record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

if len(record_set_ids) == 0:
    print("No record sets found for extraction. Unable to proceed with data extraction.")
else:
    for rs_id in record_set_ids:
        # Load records as a list of dicts
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id} with shape {df.shape}")
    # For demonstration, pick the first record set
    main_record_set_id = record_set_ids[0]
    print(f"\nFields (columns) in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll perform common EDA operations below. Choose a **numeric field** and optionally a **grouping field**, referencing both by their `@id`.

We'll filter records, normalize a numeric field, and group the data by a categorical column using the schema's `@id` references.

In [ ]:
# Example EDA on the main record set

# Inspect available columns and choose a numeric field (@id)
df_main = dataframes.get(main_record_set_id)
print(f"\nColumns in main DataFrame: {df_main.columns.tolist()}")

# Let's try to automatically select a numeric-looking column based on dtype heuristics
numeric_columns = df_main.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numeric-looking columns: {numeric_columns}")

if len(numeric_columns) == 0:
    print("No numeric columns detected. EDA cannot proceed.")
else:
    numeric_field_id = numeric_columns[0]  # use the @id
    print(f"Using numeric field: {numeric_field_id}")
    threshold = df_main[numeric_field_id].median()
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} values:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to pick a group field that's not the numeric field itself and is categorical
    possible_groups = [col for col in df_main.columns if col != numeric_field_id and df_main[col].dtype in ['object', 'category']]
    if possible_groups:
        group_field_id = possible_groups[0]
        print(f"\nGrouping by field: {group_field_id}\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field detected for grouping.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field, and (if available) its grouping by a selected categorical field.

We'll use `matplotlib` and `seaborn` for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_columns) == 0:
    print("No numeric field available for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # Visualize grouping if grouping field is available
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

- This notebook demonstrated end-to-end exploration of the FAIR² dataset using the `mlcroissant` library.
- Using the Croissant specification, all data access was performed by referencing entities via their unique `@id`.
- We loaded dataset metadata, listed record sets and fields, loaded records into DataFrames, and performed basic EDA and visualization.

For deeper research, you can explore additional fields, record sets, and relationships defined in the Croissant schema.

> **Reference:** [FAIR² Dataset on SenScience](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

---